In [ ]:
"""Day 26 / Session 3 - Lab Assignment 1: LLM API Integration - Token-Cost Estimator & Exponential-Backoff Retry."""
import time

import numpy as np

# Vendor rate cards - prices are per 1,000 tokens (the standard billing unit)
MODELS = [
    {"name": "big", "in_rate": 0.003, "out_rate": 0.015},
    {"name": "small", "in_rate": 0.001, "out_rate": 0.002},
    {"name": "huge", "in_rate": 0.010, "out_rate": 0.030},
]

TRANSIENT = {429, 500, 502, 503}


# A fake endpoint that returns HTTP-like statuses from a scripted list
def make_endpoint(statuses, body="OK"):
    state = {"n": 0}

    def call():
        i = state["n"]
        state["n"] += 1
        s = statuses[i] if i < len(statuses) else statuses[-1]
        return {"status": s, "body": body}
    call.state = state
    return call


def estimate_cost(input_tokens, output_tokens, in_rate_per_1k, out_rate_per_1k):
    return input_tokens / 1000 * in_rate_per_1k + output_tokens / 1000 * out_rate_per_1k


def estimate_batch_cost(requests, in_rate_per_1k, out_rate_per_1k):
    return sum(estimate_cost(r["input_tokens"], r["output_tokens"], in_rate_per_1k, out_rate_per_1k) for r in requests)


def cheapest_model(models, input_tokens, output_tokens):
    best_name, best_cost = None, None
    for m in models:
        cost = estimate_cost(input_tokens, output_tokens, m["in_rate"], m["out_rate"])
        if best_cost is None or cost < best_cost:               # strict <, so the first model wins ties
            best_name, best_cost = m["name"], cost
    return best_name


def backoff_delay(attempt, base=1.0, factor=2.0, cap=60.0):
    return min(cap, base * factor ** attempt)                   # attempt is 0-indexed


def call_with_retries(fn, max_attempts=5, base=1.0, factor=2.0, cap=60.0, sleep=None):
    sleep = sleep if sleep is not None else time.sleep
    for attempt in range(max_attempts):
        response = fn()
        status = response["status"]
        if status == 200:
            return response["body"]
        if status not in TRANSIENT:
            raise ValueError(f"non-retryable status {status}")  # permanent error: fail immediately, no retry
        if attempt < max_attempts - 1:
            sleep(backoff_delay(attempt, base, factor, cap))    # wait between attempts, not after the last one
    raise RuntimeError(f"gave up after {max_attempts} transient failures")
